Run all script once. The first try due to imports you might need to reset the enviroment the first time and run again the script. Use the last cell to enter the word you want and check its concreteness, imageability, physicality and precision ratings as well as an automated classification of the word's senses between BM and not BM.

#Imports
takes 15-20 minutes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install gensim
import gensim.downloader as api
nb_17 = api.load("conceptnet-numberbatch-17-06-300")
nb17_vocab = list(nb_17.key_to_index.keys())
nb17_vocab = [i.split('/')[3] for i in nb17_vocab]

w2v = api.load('word2vec-google-news-300')
w2v_vocab = list(w2v.key_to_index.keys())

[==================================================] 100.0% 1168.7/1168.7MB downloaded
[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
import pandas as pd
import numpy as np

#statistics
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr, pearsonr

In [ ]:
import nltk
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
from nltk.corpus import stopwords
from nltk.wsd import lesk

from nltk.corpus import wordnet_ic
nltk.download('wordnet_ic')
brown_ic = wordnet_ic.ic('ic-brown.dat')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet_ic to /root/nltk_data...
[nltk_data]   Unzipping corpora/wordnet_ic.zip.


In [ ]:
PATH = '/'

#Train psycholinguistic norms extender
Takes 10 minutes

In [ ]:
#read available norms
#psycholinguistic norm datasets
concreteness_df = pd.read_excel(PATH + 'psycholinguistic_data/Brisbaert_40K_Concreteness.xlsx')
physicality_df = pd.read_excel(PATH + 'psycholinguistic_data/SER_EN_5K.xls')
glasgow_df = pd.read_csv(PATH + 'psycholinguistic_data/glasgow_norms.csv')
mrc_c = pd.read_csv(PATH + 'psycholinguistic_data/MRC_corpus.csv')

#normalize row names
glasgow_df = glasgow_df[1:]
keep_columns = ['Words', 'AROU', 'VAL', 'DOM', 'IMAG', 'FAM']
glasgow_df = glasgow_df[keep_columns]
glasgow_df = glasgow_df.astype({'AROU': float, 'VAL': float, 'DOM': float, 'IMAG': float, 'FAM': float})
glasgow_df = glasgow_df.rename(columns={"Words": "Word"})

mrc_c = mrc_c.rename(columns={" word": "Word"})

In [ ]:
def get_embedding(row):
  """Gets the ConceptNet Numberbatch embedding for a word."""
  try:
    return nb_17['/c/en/' + row['Word']]
  except:
    return None

In [ ]:
# add static embeddings
for dataset_name in ['concreteness_df', 'glasgow_df', 'physicality_df']:
  dataset = globals()[dataset_name]
  dataset['nb17'] = dataset.apply(get_embedding, axis=1)

In [ ]:
train = physicality_df

max_len = max(len(x) if x is not None else 0 for x in train.nb17)
x_train = np.array([x if x is not None else np.zeros(max_len) for x in train.nb17])
y_train = train['Average SER'].values

model_physicality = SVR(kernel='rbf', C=100, gamma=0.003, epsilon=.1)
model_physicality.fit(x_train, y_train)

SVR(C=100, gamma=0.003)

In [ ]:
train = glasgow_df
max_len = max(len(x) if x is not None else 0 for x in train.nb17)
x_train = np.array([x if x is not None else np.zeros(max_len) for x in train.nb17])
y_train = train['IMAG'].values

model_imageability = SVR(kernel='rbf', C=100, gamma=0.003, epsilon=.1)
model_imageability.fit(x_train, y_train)

SVR(C=100, gamma=0.003)

In [ ]:
train = concreteness_df
max_len = max(len(x) if x is not None else 0 for x in train.nb17)
x_train = np.array([x if x is not None else np.zeros(max_len) for x in train.nb17])
y_train = train['Conc.M'].values

model_concreteness = SVR(kernel='rbf', C=100, gamma=0.003, epsilon=.1)
model_concreteness.fit(x_train, y_train)

SVR(C=100, gamma=0.003)

# Compute ratings

In [ ]:
def process_definitions (definition):
  #tokenize
  #lower
  #stopwords
  return (definition)

In [ ]:
def compute_precision (sentence, target_word): #tokenized_sentence
  def_synsets = []
  def_precission = []
  def_ic = []
  for w in sentence:
    if wordnet.synsets(w) and target_word != w:
      try:
        lesk_syn = lesk(sentence, w, synsets=wordnet.synsets(w))
        lesk_syn_tw = lesk(sentence, target_word, synsets=wordnet.synsets(target_word))
        def_synsets.append(lesk_syn)
        def_precission.append(lesk_syn.min_depth())
      except:
        continue
      try:
        def_ic.append(lesk_syn.res_similarity(lesk_syn_tw, brown_ic))
      except:
        continue
    else:
      continue
  return (def_precission, def_ic)

In [ ]:
def compute_ratings_per_word (definition, defined_word):
  ratings = {'physicality':[], 'imageability': [], 'concreteness': [],
           'nb17':[], 'precision':[], 'precision_IC':[]}
  for word in definition.split(' '):
    if word in nb17_vocab and defined_word in nb17_vocab:
      ratings['physicality'].append(model_physicality.predict(nb_17['/c/en/' + word].reshape(1, -1))[0])
      ratings['imageability'].append(model_imageability.predict(nb_17['/c/en/' + word].reshape(1, -1))[0])
      ratings['concreteness'].append(model_concreteness.predict(nb_17['/c/en/' + word].reshape(1, -1))[0])
      ratings['nb17'].append(nb_17.similarity('/c/en/'+ word.lower(), '/c/en/'+ defined_word.lower()))
    else: #mejor continue o .append(None)?
      continue
  ratings['precision'] = compute_precision(word_tokenize(definition), defined_word)[0]
  ratings['precision_IC'] = compute_precision(word_tokenize(definition), defined_word)[1]
  return ratings

In [ ]:
def compute_mean_max_min (ratings_per_word):
  mean_ratings = {}
  max_ratings = {}
  min_ratings = {}

  for key in ratings_per_word.keys():
    if ratings_per_word[key]:
        max_ratings.update({key:max(ratings_per_word[key])})
        min_ratings.update({key:min(ratings_per_word[key])})
        mean_ratings.update({key:np.mean(ratings_per_word[key])})
    else:
        continue

  return {'mean': mean_ratings, 'max': max_ratings, 'min': min_ratings}

# Utils

In [ ]:
def get_definitions_with_ratings (word):
  definitions = []
  for sense in wordnet.synsets(word):
    ratings_per_word = compute_ratings_per_word(sense.definition(), word)
    definitions.append({'name':sense.name(),
                        'definition': sense.definition(),
                        #'ratings_per_word': ratings_per_word,
                        'mean_ratings': compute_mean_max_min(ratings_per_word)['mean'],
                        'max_ratings': compute_mean_max_min(ratings_per_word)['max'],
                        'min_ratings': compute_mean_max_min(ratings_per_word)['min']
                        })
  return definitions